# PLANNER - EXECUTOR - CRITIC AGENT

In [1]:
from langgraph.graph import StateGraph, START, MessagesState, add_messages, END
from operator import add
from langchain.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage, RemoveMessage
from typing import Annotated
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, DeclarativeBase
from dotenv import load_dotenv
from sqlalchemy import text
from sqlalchemy.orm import Session
import os
from langchain.tools import tool

### Tools

In [2]:
from dotenv import load_dotenv
from langchain_tavily import TavilySearch
load_dotenv()

tavily_search_tool = TavilySearch(
    max_results=2
)

### Agent State

In [3]:
class AgentState(MessagesState):
    user_question: str
    tasks: list
    results: Annotated[list, add]
    current_task_idx: int
    is_user_question_valid: bool
    response_accuracy: int
    is_replanning_required: bool
    replanning_feedback: str
    final_curated_response: str
    task_count: int
    is_executor_response_review_needed: bool
    executor_reviewer_feedback: str

### LLM

In [4]:
from langchain_openai import ChatOpenAI

def get_llm_instance(model: str="gpt-4o-mini"):
    model = ChatOpenAI(
        model=model,
        temperature=0.2,
        max_tokens=None,
        timeout=None,
        max_retries=2
    )
    return model

### Planner Node

In [5]:
from pydantic import BaseModel

class Task(BaseModel):
    id: int
    instructions: str

class PlannerResponse(BaseModel):
    tasks: list[Task]

In [6]:
import json

async def planner_node(state: AgentState):
    if state.get('is_replanning_required') is None:
        PLANNER_PROMPT = (
        "You are a Planning Agent whose task is to split a larger task into sub tasks.\n"
        "Combine multiple tasks into one if you feel they can be answered in one go.\n"
        "Each sub task must have a unique id and a short and complete instructions guiding the executor agent to perform the task.\n"
        "Make sure that a task is not dependent on future / next tasks results.\n"
        "For simple and generic questions create very few tasks that clearly answer the question.\n"
        "Don't generate unnecessary tasks that are out of scope of user query.\n"
        "Don't generate tasks to review response as there is a reviewer present.\n"
        "Executor agent that executes the sub task will have only a web search tool available. Keep this in mind while creating sub tasks.\n\n"
        "USER QUESTION:\n"
        f"{state['user_question']}"
    )
    else:
        # Replanning
        previous_task_results = ''
        for n in range(len(state.get('results', []))):
            n_result = state['results'][n] 
            task_id = n_result.get('id', "")
            task_instructions = n_result.get('instructions', "")
            task_result = n_result.get('result', "")
            previous_task_results += f"Task (ID: {task_id}):\n{task_instructions}\nTask (ID: {task_id}) Result:\n{task_result}\n\n"
        
        PREVIOUS_RESULTS_PROMPT = (
            "PREVIOUS GENERATED TASKS AND RESULTS:\n"
            f"{previous_task_results}"
            "\n\n"
        )
        
        PLANNER_PROMPT = (
        "You are a Planning Agent whose task is to split a larger task into sub tasks.\n"
        "You earlier generated a plan. the tasks in the plan were executed and the results were reviewed by a human. It was found that tasks are insufficient to fully answer the user question.\n"
        "Based on the feedback you need to generate a revised plan containing ONLY THE ADDITIONAL TASKS REQUIRED TO FULLY ANSWER USER QUESTION WITHOUT REPEATING THE PREVIOUSLY GENERATED TASKS.\n"
        "The new tasks must be a continuation of previous generated tasks.\n"
        f"START THE ID OF THE NEW TASKS FROM {len(state.get('results', 1)) + 1} AS A CONTINUATION OF PREVIOUS TASKS."
        "\n\n"
        f"{PREVIOUS_RESULTS_PROMPT}"
        "\n\n"
        "INSTRUCTIOINS:\n"
        "Each sub task must have a unique id and a short and complete instructions guiding the executor agent to perform the task.\n"
        "Combine multiple tasks into one if you feel they can be answered in one go.\n"
        "Make sure that a task is not dependent on future/next tasks results.\n"
        "Don't generate unnecessary tasks that are out of scope of user query.\n"
        "Don't generate tasks to review response as there is a reviewer present.\n"
        "Executor agent that executes the sub task will have only a web search tool available. Keep this in mind while creating sub tasks.\n\n"
        "USER QUESTION:\n"
        f"{state['user_question']}"
    )
        
    llm = get_llm_instance()
    planner_llm = llm.with_structured_output(PlannerResponse)
    
    max_retries = 3

    for attempt in range(max_retries + 1):
        response = await planner_llm.ainvoke(PLANNER_PROMPT)

        if not response.model_dump_json():
            response_json = {}
            break

        try:
            response_json = json.loads(response.model_dump_json())
            if "tasks" not in response_json:
                raise JSONDecodeError("Invalid json")
            break

        except json.JSONDecodeError:
            if attempt == max_retries:
                response_json = dict()

    return {"messages": [AIMessage(content=response_json.get('tasks', []), name='planner', type='ai')],
        "tasks": response_json.get('tasks', []), "current_task_idx": 0, 'task_count': len(response_json.get('tasks', []))}

### Supervisor Node

In [7]:
import json
from datetime import datetime

async def supervisor_node(state: AgentState):
    
    if state.get("task_count", 0) == state.get("current_task_idx", 0) and state.get("current_task_idx", 0) > 0:
        return {'messages': [AIMessage(content='All tasks are completed successfully', id=uuid4().hex, name='Supervisor', type='ai')]}

    
    messages = []
    if state.get('messages', []):
        messages = [RemoveMessage(id=m.id) for m in state['messages']]
    
    previous_task_results = ''
    for n in range(len(state.get('results', []))):
        n_result = state['results'][n]
        task_id = n_result.get('id', "")
        task_instructions = n_result.get('instructions', "")
        task_result = n_result.get('result', "")
        previous_task_results += f"Previous Task {task_id}:\n{task_instructions}\nPrevious Task {task_id} Result:\n{task_result}\n\n"
    
    PREVIOUS_RESULTS_PROMPT = (
        "Refer to the previous task details provided to solve the task.\n"
        "PREVIOUS TASK DETAILS:\n"
        f"{previous_task_results}"
        "\n\n"
    )

    
    current_task_idx = state.get('current_task_idx', 0)
    newline = '\n'
    current_date_time = str(datetime.now())
    current_task_instructions = state['tasks'][current_task_idx]['instructions']
    PROMPT = (
        "You are assigned a task to solve which is part of a group of multiple tasks.\n"
        "A human reviewer reviews your task result and gives a feedback.\n"
        "In case you find any reviewer feedback in the conversation, follow it and take corrective steps to generate an accurate response.\n"
        f"Current date and time (YYYY-MM-DD HH:MM:SS.ms format) is {current_date_time}."
        "\n"
        "RELY ON YOUR INTERNAL KNOWLEDGE WHEN WEB SEARCH IS NOT REQUIRED. IF YOU RELY ON RESULTS FOUND IN WEB SEARCH, YOUR ANSWERS MUST BE GROUNDED TO WEB SEARCH RESULTS.\n"
        "In case of multiple web search results, consider most recent one.\n"
        "DO NOT MAKE DUPLICATE TOOL CALLS, IF A TOOL CALL IS PRESENT IN THE CONVERSATION HISTORY, USE IT INSTEAD OF CALLING TOOL ONCE AGAIN.\n"
        "IF PREVIUOS TASK RESULTS CONTAIN AN ANSWER TO THE CURRENT TASK THEN USE THEM INSTEAD OF MAKING A TOOL CALL.\n"
        "**If your task contains some private data that is not present on the web, then return a response that data is not publicly available.**\n"
        "The response should be short and complete in less than 3 sentences. It should solve the task without providing additional information that is not necessary.\n"
        "Include a detailed reason for your response so that the human reviewer understands your appraoch.\n"
        f"{'Few tasks in the group are already solved.' + newline  if previous_task_results else ''}"
        f"{PREVIOUS_RESULTS_PROMPT if previous_task_results else ''}"
        f"YOUR TASK INSTRUCTIONS:\n{current_task_instructions}\n\n"
        
        f"{'ADDITIONAL IMPORTANT INSTRUCTIONS:' + newline + '- ONLY SOLVE THE TASK ASSIGNED TO YOU.' if previous_task_results else ''}"
    )
    messages.append(HumanMessage(content=PROMPT, id=uuid4().hex, name='human', type='human'))
    
    return {"messages": messages}

### Executor Node

In [8]:
import json

async def executor_node(state: AgentState):
    llm = get_llm_instance()
    TOOLS = [tavily_search_tool]
    executor_llm = llm.bind_tools(tools=TOOLS, strict=False)
    
    response = await executor_llm.ainvoke(state['messages'])
    return {"messages": [response]}

## Reviewer Node

In [9]:
class ReviewerResponse(BaseModel):
    is_executor_response_review_needed: bool
    reviewer_feedback: str

In [10]:
async def get_tool_calls_made_by_agent(state):
    tool_calls_messages = []
    for message in state['messages']:
        if isinstance(message, ToolMessage):
            tool_calls_messages.append(message)
    return tool_calls_messages

In [11]:
import json

async def reviewer_node(state: AgentState):
    current_task_idx = state.get("current_task_idx", 0)
    current_task_id = state['tasks'][current_task_idx].get('id', 0)
    current_task_instructions = state['tasks'][current_task_idx].get('instructions', '')
    current_task_result = state['messages'][-1].content
    tool_calls_messages = await get_tool_calls_made_by_agent(state)

    previous_task_results = ''
    for n in range(len(state.get('results', []))):
        n_result = state['results'][n]
        task_id = n_result.get('id', "")
        task_instructions = n_result.get('instructions', "")
        task_result = n_result.get('result', "")
        previous_task_results += f"Previous Task {task_id}:\n{task_instructions}\nPrevious Task {task_id} Result:\n{task_result}\n\n"
    
    PREVIOUS_RESULTS_PROMPT = (
        "PREVIOUS TASK DETAILS:\n"
        f"{previous_task_results}"
        "\n\n"
    )
    
    REVIEWER_PROMPT = (
    "You are a Reviewer/Critic Agent whose task is to review an AI Agent's response.\n"
    "A larger task is split into multiple sub tasks and an AI agent solves each task using information from previous task results.\n"
    "Given the previous task results (if any), current task results and tool calls made by agent to solve current tasks.\n"
    "You need to decide if the current task response fully solves current task.\n"
    "Look at the reason for the agent's response while taking a decision.\n"
    "The AI agent has access to web search tool only.\n"
    "If there are any tool calls made by the agent, ensure the response is grounded to the tool response.\n"
    "If tool calls are present, you need to rely on information present in the tool result not your internal knowledge."
    "In case of no tool calls, recommend one, if needed.\n"
    "**IF PREVIUOS TASK RESULTS CONTAIN AN ANSWER TO THE CURRENT TASK THEN USE THEM INSTEAD OF MAKING A TOOL CALL, THEN YOU SHOULD NOT RECOMMEND TOOL CALL.**\n"
    "In case of multiple web search results, consider most recent one by date.\n"
    "If the agent's current task response needs review, give a feedback mentioning the review reason and corrective actions.\n"
    "YOU NEED TO REVIEW ONLY THE CURRENT TASK RESULTS.\n\n"
    f"{PREVIOUS_RESULTS_PROMPT if previous_task_results else ''}"
    "CURRENT TASK:\n"
    f"{current_task_instructions}"
    "\nCURRENT TASK RESULT:\n"
    f"{current_task_result}"
    "\n"
    "CURRENT TASK TOOL CALLS:\n"
    f"{tool_calls_messages}"
)
        
    llm = get_llm_instance(model='gpt-5-mini')
    reviewer_llm = llm.with_structured_output(ReviewerResponse)
    
    max_retries = 3

    for attempt in range(max_retries + 1):
        response = await reviewer_llm.ainvoke(REVIEWER_PROMPT)

        if not response.model_dump_json():
            response_json = {}
            break

        try:
            response_json = json.loads(response.model_dump_json())
            if "is_executor_response_review_needed" not in response_json:
                raise JSONDecodeError("Invalid json")
            break

        except json.JSONDecodeError:
            if attempt == max_retries:
                response_json = dict()

    if response_json['is_executor_response_review_needed']:    
        return {"messages": [HumanMessage(content=json.dumps(response_json), name='reviewer', type='human')], 
               'is_executor_response_review_needed': response_json['is_executor_response_review_needed']}
    else:
        return { 
                "messages": [HumanMessage(content=json.dumps(response_json), name='reviewer', type='human')],
                "results": [{"id": current_task_id, "instructions": current_task_instructions, "result": current_task_result}], 
                "current_task_idx": state.get("current_task_idx", 0) + 1,
                'is_executor_response_review_needed': response_json['is_executor_response_review_needed']
               }   

### Synthesizer Node

In [12]:
class SynthesizerResponse(BaseModel):
    response_accuracy: int
    is_replanning_required: bool
    replanning_feedback: str
    final_curated_response: str

In [13]:
import pandas as pd
from IPython.display import display

async def synthesizer_node(state: AgentState):
    previous_task_results = ''
    for n in range(len(state.get('results', []))):
        n_result = state['results'][n]
        task_id = n_result.get('id', "")
        task_instructions = n_result.get('instructions', "")
        task_result = n_result.get('result', "")
        previous_task_results += f"Task {task_id}:\n{task_instructions}\nTask {task_id} Result:\n{task_result}\n\n"
    
    PREVIOUS_RESULTS_PROMPT = (
        "SUB TASK DETAILS:\n"
        f"{previous_task_results}"
        "\n\n"
    )
    SYNTHESIZER_PROMPT = (
    "You are a response synthesizer who curates final response to a user question by combining results of sub tasks.\n"
    "A Planner AI Agent has split the user question into multiple sub tasks.\n"
    "You need to validate if the sub tasks completely answer the user question or not.\n"
    "**Don't validate the task responses, just validate whether the plan covers all tasks required to answer the question.**\n"
    "If the task result says information is private and not able to found. Respect it. Don't order for plan review.\n"
    "If the tasks created are incomplete, you need to generate a corrective feedback to the Planner AI Agent which is used for replanning and final curated response must be an empty string.\n"
    "If the tasks created are enough to fully answer the user question, then generate the final curated response to the user question based on the sub task results provided and replanning feedback must be an empty string.\n"
    "If the task results mention that answer or data is not found, mention the same in final response, don't ask for replan.\n"
    "Provide a final curated response accuracy score based on the user question between 0 to 100 like 90, 95, etc.\n\n"
    "USER QUESTION:\n"
    f"{state['user_question']}\n\n"
    f"{PREVIOUS_RESULTS_PROMPT}"
)
    llm = get_llm_instance()
    synthesizer_llm = llm.with_structured_output(SynthesizerResponse)

    max_retries = 3

    for attempt in range(max_retries + 1):
        response = await synthesizer_llm.ainvoke(SYNTHESIZER_PROMPT)

        if not response.model_dump_json():
            response_json = {}
            break

        try:
            response_json = json.loads(response.model_dump_json())
            break

        except json.JSONDecodeError:
            if attempt == max_retries:
                response_json = dict()


    newline = '\n'
    return {'messages': [AIMessage(content=f"{'Agent Execution Completed Successfully.' + (newline * 2) + 'FINAL RESPONSE (Accuracy: ' + str(response_json.get('response_accuracy', 0)) +'):' + newline + response_json.get('final_curated_response', '')}")
                         if not response_json.get('is_replanning_required', False) else 
                         AIMessage(content=f"{'Plan is not accurate. Replanning Required.' + (newline * 2) + 'PLAN FEEDBACK:' + newline + response_json.get('replanning_feedback', '')}")], 
           "response_accuracy": response_json.get('response_accuracy', 0), 
           'is_replanning_required': response_json.get('is_replanning_required', False),
           'replanning_feedback': response_json.get('replanning_feedback', ''),
           'final_curated_response': response_json.get('final_curated_response', '')
           }

In [14]:
from typing import Literal

async def executor_conditional_edge(state: AgentState) -> Literal["executor", "synthesizer"]:
    if state.get("task_count", 0) == state.get("current_task_idx", 0) and state.get("current_task_idx", 0) > 0:
        return "synthesizer"
    else:
        return "executor"

In [15]:
async def tool_conditional_edge(state: AgentState) -> Literal["tools", 'reviewer']:
    if isinstance(state['messages'][-1], AIMessage) and state['messages'][-1].tool_calls:
        return "tools"
    else:
        return "reviewer"

In [16]:
async def replanning_conditional_edge(state: AgentState) -> Literal["planner", END]:
    if state['is_replanning_required']:
        return "planner"
    else:
        return END

In [17]:
async def reviewer_conditional_edge(state: AgentState) -> Literal["executor", "supervisor"]:
    if state['is_executor_response_review_needed']:
        return "executor"
    else:
        return "supervisor"

In [18]:
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import tools_condition, ToolNode

async def get_planner_executor_agent():
    planner_executor_agent = StateGraph(AgentState)
    planner_executor_agent.add_node("planner", planner_node)
    planner_executor_agent.add_node("supervisor", supervisor_node)
    planner_executor_agent.add_node("executor", executor_node)
    planner_executor_agent.add_node("reviewer", reviewer_node)
    planner_executor_agent.add_node("tools", ToolNode(tools=[tavily_search_tool], handle_tool_errors=True))
    planner_executor_agent.add_node("synthesizer", synthesizer_node)    
    
    planner_executor_agent.add_edge(START, 'planner')
    planner_executor_agent.add_edge('planner', 'supervisor')
    planner_executor_agent.add_conditional_edges("supervisor", executor_conditional_edge)
    planner_executor_agent.add_conditional_edges("executor", tool_conditional_edge)
    planner_executor_agent.add_conditional_edges("reviewer", reviewer_conditional_edge)
    planner_executor_agent.add_edge("tools", "executor")
    planner_executor_agent.add_conditional_edges("synthesizer", replanning_conditional_edge) 
    
    planner_executor_agent = planner_executor_agent.compile()
    return planner_executor_agent


In [19]:
# (await get_planner_executor_agent()).get_graph(xray=1).draw_mermaid_png(output_file_path="graph.png")

In [20]:
async def get_response_streamer(user_query, run_id):
    planner_executor_agent = await get_planner_executor_agent()
    response_streamer = planner_executor_agent.astream(
            {
                "user_question": user_query
            },
            stream_mode="updates",
            config={"configurable": {"thread_id": run_id}}
        )
    return response_streamer

In [21]:
from uuid import uuid4
query = '''
what is the major business activity and what's recent percent decline in its stock of the comapny whose ceo resigned recently due to a scandal.
what does the other person involved in scandal do and what is the person's nationality.
'''
response_streamer = await get_response_streamer(query, uuid4().hex)

In [22]:
async for chunk in response_streamer:
    for node_name, node_output in chunk.items():
        print(f"\n--- Node: {node_name} ---")
        if "messages" in node_output:
            for msg in node_output["messages"]:
                msg.pretty_print()


--- Node: planner ---
================================== Ai Message ==================================
Name: planner

[{'id': 1, 'instructions': 'Identify the company whose CEO recently resigned due to a scandal. Provide the name of the company.'}, {'id': 2, 'instructions': 'Find out the major business activity of the company identified in Task 1.'}, {'id': 3, 'instructions': 'Determine the recent percentage decline in the stock price of the company identified in Task 1.'}, {'id': 4, 'instructions': "Identify the other person involved in the scandal related to the CEO's resignation. Provide their name and role."}, {'id': 5, 'instructions': 'Find out the nationality of the person identified in Task 4.'}]

--- Node: supervisor ---
================================ Remove Message ================================


================================ Human Message =================================
Name: human

You are assigned a task to solve which is part of a group of multiple tasks.
A huma